<a href="https://colab.research.google.com/github/NathanaelLUCETTE/projet_ML/blob/main/pokemon_clustering_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Project Pokemon Clustering**

##**Rapport - Steps**

###**Introduction**
The objective of this project is to investigate whether competitive roles commonly used in the Pokémon community—such as fast pivot, wallbreaker, bulky tank, or setup sweeper—can be discovered automatically through unsupervised learning.
These roles are traditionally defined by expert players (e.g., *Smogon*), but they are not explicitly labeled in the dataset.
This makes clustering an appropriate approach to uncover latent structure based solely on Pokémon attributes.

Using the Ultimate Pokémon Dataset 2025, which compiles statistical, physical, and categorical information for all official Pokémon, we aim to determine whether meaningful groups emerge naturally from the data and how these groups relate to known competitive archetypes.

###**Dataset Selection**
The dataset used in this project is the Ultimate Pokémon Dataset 2025, sourced from Kaggle.
It includes Pokémon from all generations and provides a wide range of features:

- Base stats: HP, Attack, Defense, Special Attack, Special Defense, Speed

- Derived attributes: total stats, offensive/defensive totals

- Physical attributes: height, weight, BMI

- Categorical attributes: primary/secondary type, color, shape, habitat

- Rarity indicators: legendary, mythical, baby

- Flavor data: descriptions, abilities, egg groups (not all used in clustering)

This dataset was selected because it is comprehensive, up‑to‑date, and structured in a way that facilitates both exploratory analysis and machine learning workflows.

###Dataset Description
The dataset contains over 1,000 Pokémon entries, each described by dozens of attributes.
The features can be grouped into three main categories:





####Statistical Features
These represent the core combat capabilities of each Pokémon:

- HP

- Attack

- Defense

- Special Attack

- Special Defense

- Speed

- Total base stats

These variables are essential for identifying offensive and defensive roles.

####Physical Features
These include:

- Height

- Weight

- BMI (computed)

Physical attributes may correlate with defensive roles (e.g., heavy tanks) or offensive archetypes.

####Categorical Features
These include:

- Type 1 and Type 2

- Color

- Shape

- Habitat

- Growth rate

- Egg groups

These features require encoding before being used in clustering.

###***À modifier***
4. Problem Definition
The central question of this project is:

Can we automatically identify meaningful Pokémon roles using unsupervised learning, without any predefined labels?

More specifically, we aim to:

Determine whether Pokémon naturally form clusters based on their attributes

Compare these clusters to known competitive roles

Evaluate how different feature subsets influence cluster formation

Test the generalization of the clustering model on new or fan‑made Pokémon

This is a clustering problem, where the number and nature of the clusters are not known in advance.

5. Methodology
The study follows a structured machine learning workflow:

5.1. Exploratory Data Analysis (EDA)
Goals:
Understand distributions of base stats

Identify correlations

Detect outliers (e.g., legendaries, ultra-fast Pokémon)

Explore type frequencies and physical characteristics

Tools:
Histograms

Boxplots

Correlation matrices

PCA visualizations

5.2. Data Preparation
Feature Selection
Different subsets of features are used depending on the clustering objective:

-  Objective	Features
-  Offensive roles	Attack, SpA, Speed
-  Defensive roles	HP, Defense, SpD, BMI
-  Mixed roles	All base stats + physical attributes
-  Pure statistical clustering	Base stats only
-  Extended clustering	Stats + one‑hot encoded types


Encoding
- One‑hot encoding for categorical variables

- Standardization (z‑score) for numerical variables

- Cleaning
- Removing unused columns (sprite URL, description, abilities)

- Handling missing values

5.3. Determining the Optimal Number of Clusters
We use several metrics:

-  Elbow Method

- Silhouette Score

- Davies–Bouldin Index

- Visual inspection of PCA projections

This step is repeated for each feature subset.

5.4. Clustering Algorithms
We apply multiple algorithms to compare results:

- K‑Means
-  Baseline method

- Easy to interpret via centroids

- Agglomerative Clustering
- Produces a dendrogram

- Useful for hierarchical relationships

- DBSCAN
- Detects outliers (e.g., legendaries)

Finds clusters of varying density

5.5. Cluster Interpretation
For each cluster, we analyze:

-  Mean and variance of stats

- Dominant types

- Physical characteristics

- Comparison with competitive roles (Smogon)

 - Examples:

    - High Speed + high Attack → fast sweeper

    - High HP + high Defense → physical wall

    -  High Speed + pivoting stats → fast switch/pivot


5.6. External Validation
Generation IX Pokémon
Projected into the cluster space

- Used to test generalization

Fan‑made Pokémon (Fakemon)
Added to evaluate robustness

Useful for dataset enrichment

6. Expected Outcomes
We expect to observe:

Distinct offensive and defensive clusters

A cluster dominated by legendary Pokémon

Mixed‑role clusters (e.g., bulky attackers)

Clusters that align with Smogon roles

7. Conclusion (Preview)
This project aims to demonstrate that competitive Pokémon roles can emerge naturally from statistical and physical attributes through unsupervised learning.
The results will highlight the strengths and limitations of clustering for modeling complex strategic roles.

##Importation du dataset et affichage

###Importation

###Importation depuis drive (si problème avec kaggle)

In [16]:
#Vérification de l'emplacement du fichier
!ls drive/MyDrive/Sem_etranger/Project_MachineLearning

#Importation depuis le drive
import pandas as pd
df = pd.read_csv("drive/MyDrive/Sem_etranger/Project_MachineLearning/pokemon_database.csv")




pokemon_clustering_project.ipynb  pokemon_database.csv


###Importation avec téléversement et kaggle

In [18]:
# Install Kaggle if needed
!pip install kaggle

# Create the .kaggle directory
!mkdir -p ~/.kaggle

# Upload your kaggle.json (each teammate must upload their own)
from google.colab import files
files.upload()

# Move kaggle.json to the correct location
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the dataset
!kaggle datasets download -d darkmatternet/ultimate-pokmon-dataset-2025

# Unzip the dataset
!unzip ultimate-pokmon-dataset-2025.zip

# Load the dataset
import pandas as pd
df = pd.read_csv("pokemon_dataset.csv")
df.head()


Saving pokemon_dataset.csv to pokemon_dataset (1).csv
mv: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/darkmatternet/ultimate-pokmon-dataset-2025
License(s): CC0-1.0
ultimate-pokmon-dataset-2025.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  ultimate-pokmon-dataset-2025.zip
replace pokemon_complete_2025.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: pokemon_complete_2025.csv  


,pokedex_id,name,genus,generation,type_1,type_2,num_types,hp,attack,defense,...,sprite_url,is_dual_type,bmi,attack_defense_ratio,physical_total,special_total,offensive_total,defensive_total,gender_distribution,stat_tier
0,1,bulbasaur,Seed Pokémon,I,grass,poison,2,45,49,49,...,https://raw.githubusercontent.com/PokeAPI/spri...,True,14.1,1.00,98,130,159,159,88% Male / 12% Female,Below Average (300-399)
1,2,ivysaur,Seed Pokémon,I,grass,poison,2,60,62,63,...,https://raw.githubusercontent.com/PokeAPI/spri...,True,13.0,0.98,125,160,202,203,88% Male / 12% Female,Average (400-499)
2,3,venusaur,Seed Pokémon,I,grass,poison,2,80,82,83,...,https://raw.githubusercontent.com/PokeAPI/spri...,True,25.0,0.99,165,200,262,263,88% Male / 12% Female,Strong (500-599)
3,4,charmander,Lizard Pokémon,I,fire,NaN,1,39,52,43,...,https://raw.githubusercontent.com/PokeAPI/spri...,False,23.6,1.21,95,110,177,132,88% Male / 12% Female,Below Average (300-399)
4,5,charmeleon,Flame Pokémon,I,fire,NaN,1,58,64,58,...,https://raw.githubusercontent.com/PokeAPI/spri...,False,15.7,1.10,122,145,224,181,88% Male / 12% Female,Average (400-499)


###Tests

In [ ]:
cols_to_drop = [
    "num_types", "base_experience", "ability_1", "ability_2", "genus",
    "hidden_ability", "habitat", "growth_rate", "egg_groups", "gender_rate",
    "is_mythical", "sprite_url", "description", "gender_distribution", "bmi", "base_happiness"
]

df.drop(columns=cols_to_drop)


,pokedex_id,name,generation,type_1,type_2,hp,attack,defense,sp_attack,sp_defense,...,capture_rate,base_happiness,hatch_counter,is_dual_type,attack_defense_ratio,physical_total,special_total,offensive_total,defensive_total,stat_tier
0,1,bulbasaur,I,grass,poison,45,49,49,65,65,...,45,70,20,True,1.00,98,130,159,159,Below Average (300-399)
1,2,ivysaur,I,grass,poison,60,62,63,80,80,...,45,70,20,True,0.98,125,160,202,203,Average (400-499)
2,3,venusaur,I,grass,poison,80,82,83,100,100,...,45,70,20,True,0.99,165,200,262,263,Strong (500-599)
3,4,charmander,I,fire,NaN,39,52,43,60,50,...,45,70,20,False,1.21,95,110,177,132,Below Average (300-399)
4,5,charmeleon,I,fire,NaN,58,64,58,80,65,...,45,70,20,False,1.10,122,145,224,181,Average (400-499)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1020,1021,raging-bolt,IX,electric,dragon,125,73,91,137,89,...,10,0,50,True,0.80,164,226,285,305,Strong (500-599)
1021,1022,iron-boulder,IX,rock,psychic,90,120,80,68,108,...,10,0,50,True,1.50,200,176,312,278,Strong (500-599)
1022,1023,iron-crown,IX,steel,psychic,90,72,100,122,108,...,10,0,50,True,0.72,172,230,292,298,Strong (500-599)
1023,1024,terapagos,IX,normal,NaN,90,65,85,65,85,...,255,50,5,False,0.76,150,150,190,260,Average (400-499)


####Test d'affichage

In [ ]:
##Importation matplotlib, numpy, pandas
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

#Chargement du dataset dans pandas
import pandas as pd

df = pd.read_csv("pokemon_database.csv")

attacks = df["attack"]
defense = df["defense"]
hp = df["hp"]
speed = df["speed"]
spA = df["sp_attack"]
spD = df["sp_defense"]

plt.scatter(attacks, defense)
plt.title("Graph_2D_attack/def")
plt.xlabel("Attack stat")
plt.ylabel("Def stat")
plt.show()

plt.scatter(speed, hp)
plt.title("Graph_2D_speed/hp")
plt.xlabel("Speed stat")
plt.ylabel("HP stat")
plt.show()

plt.scatter(spA, spD)
plt.title("Graph_2D_spA/spD")
plt.show()



FileNotFoundError: [Errno 2] No such file or directory: 'pokemon_complete_2025.csv'

##**EDA - Exploratory Data Analysis**

###Matrix of correlation

In [33]:
#Chargement du dataset dans pandas
import pandas as pd
df = pd.read_csv("pokemon_dataset.csv")

cols_to_drop = [
    "generation", "name", "type_1", "type_2", "genus", "hidden_ability", "is_legendary", "is_mythical", "sprite_url", "description", "gender_distribution", "bmi",
    "ability_1", "ability_2", "egg_groups", "habitat", "is_baby", "is_dual_type", "stat_tier", "growth_rate", "shape", "color"
]

df_num = df.drop(columns=cols_to_drop)
df_num.head()


##Selection of the numerical features (HP, attacks, speed...)
corr = df_num.corr()
corr

,pokedex_id,num_types,hp,attack,defense,sp_attack,sp_defense,speed,base_stat_total,height_m,...,base_experience,capture_rate,base_happiness,hatch_counter,gender_rate,attack_defense_ratio,physical_total,special_total,offensive_total,defensive_total
pokedex_id,1.000000,0.109656,0.147574,0.173065,0.134994,0.118280,0.100935,0.092647,0.194158,0.076217,...,0.207070,-0.126396,-0.424879,0.187867,-0.143916,0.001666,0.180120,0.127335,0.170494,0.166089
num_types,0.109656,1.000000,0.095403,0.126056,0.172577,0.174845,0.154291,0.116012,0.212613,0.118115,...,0.209031,-0.168069,-0.133860,0.059854,-0.052625,-0.033029,0.174200,0.191014,0.184777,0.183883
hp,0.147574,0.095403,1.000000,0.476141,0.299013,0.359616,0.369087,0.179877,0.667127,0.482714,...,0.682954,-0.479332,-0.171690,0.344525,-0.164157,0.087878,0.453625,0.421198,0.451904,0.710395
attack,0.173065,0.126056,0.476141,1.000000,0.465565,0.281880,0.226126,0.351055,0.714396,0.383843,...,0.621429,-0.520345,-0.301312,0.334810,-0.285025,0.464227,0.858635,0.295567,0.726133,0.508032
defense,0.134994,0.172577,0.299013,0.465565,1.000000,0.210107,0.503281,0.008047,0.629466,0.347968,...,0.552585,-0.456910,-0.227007,0.275093,-0.248709,-0.453791,0.853397,0.403539,0.306092,0.795802
sp_attack,0.118280,0.174845,0.359616,0.281880,0.210107,1.000000,0.493034,0.423150,0.701154,0.318759,...,0.645135,-0.524140,-0.239051,0.398240,-0.311379,0.016188,0.287714,0.879316,0.755965,0.453449
sp_defense,0.100935,0.154291,0.369087,0.226126,0.503281,0.493034,1.000000,0.214105,0.697998,0.275874,...,0.656577,-0.508516,-0.161817,0.313583,-0.240217,-0.283808,0.424678,0.847865,0.414167,0.804458
speed,0.092647,0.116012,0.179877,0.351055,0.008047,0.423150,0.214105,1.000000,0.553797,0.201195,...,0.514709,-0.403098,-0.228695,0.310395,-0.249058,0.276938,0.211427,0.375081,0.778812,0.168557
base_stat_total,0.194158,0.212613,0.667127,0.714396,0.629466,0.701154,0.697998,0.553797,1.000000,0.505971,...,0.924748,-0.729900,-0.338408,0.499080,-0.380691,0.033240,0.785347,0.809386,0.872960,0.860817
height_m,0.076217,0.118115,0.482714,0.383843,0.347968,0.318759,0.275874,0.201195,0.505971,1.000000,...,0.477108,-0.301055,-0.313241,0.357254,-0.195229,0.030597,0.427617,0.345274,0.401485,0.477496
